## Download data

In [1]:
%%sh
if [ ! -f flowers-recognition.zip ] 
then
   kaggle datasets download -d alxmamaev/flowers-recognition
fi
if [ ! -d flowers ] 
then
   unzip flowers-recognition.zip >/dev/null
fi

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset,DataLoader,random_split
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import datetime
import copy

In [3]:
seed=9797 
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True


### Create custom dataset class

- Since we use ```random_split``` we cannot use different transforms to the training and validation datasets
- Dataset is an abstract class. It cannot be instantiated 
- We need to wrap the datasets obtained from ```random_split``` with a custom dataset class

In [4]:
class MyDataset(Dataset):
    def __init__(self,subset,transform=None):
        self.subset=subset
        self.transform=transform
    def __getitem__(self,idx):
        x,y=self.subset[idx]
        if self.transform:
            x=self.transform(x)
        return x,y
    def __len__(self):
        return len(self.subset)

#### Data augmentation


In [5]:
augment_choices={'trivial':transforms.TrivialAugmentWide(),'random':transforms.RandAugment(),
                 'mix':transforms.AugMix(),'auto':transforms.AutoAugment()}
aug='trivial'

In [6]:
data_transforms = {
     'train': 
        transforms.Compose([
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0., 0., 0.], [1., 1., 1.])
    ]) if aug=='None' else
     
     transforms.Compose([
        transforms.TrivialAugmentWide(),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0., 0., 0.], [1., 1., 1.])
    ]), 
  
    'val': transforms.Compose([
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0., 0., 0.], [1., 1., 1.])
    ])
}



- Why did we choose 224?

In [ ]:
dataset=torchvision.datasets.ImageFolder("flowers")
class_names=dataset.classes
itr=iter(dataset)
fig=plt.figure()
fig.tight_layout()
plt.subplots_adjust( wspace=1, hspace=1)
for i in range(20):
            img,label=next(itr)
            t=fig.add_subplot(4,5,i+1)
            # set the title of the image equal to its label
            t.set_title(str(label))
            t.axes.get_xaxis().set_visible(False)
            t.axes.get_yaxis().set_visible(False)
            plt.imshow(img)

In [7]:

train_d,valid_d=random_split(dataset,lengths=[0.8,0.2])

datasets={'train':train_d,'val':valid_d}
image_datasets = {x: MyDataset(datasets[x],data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32,
                                             shuffle=True, num_workers=2)
              for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#### What models are available?

In [8]:
list_of_models=models.list_models()
[(m,list_of_models.index(m)) for m in list_of_models if 'resnet' in m]

[('deeplabv3_resnet101', 6),
 ('deeplabv3_resnet50', 7),
 ('fasterrcnn_resnet50_fpn', 25),
 ('fasterrcnn_resnet50_fpn_v2', 26),
 ('fcn_resnet101', 27),
 ('fcn_resnet50', 28),
 ('fcos_resnet50_fpn', 29),
 ('keypointrcnn_resnet50_fpn', 32),
 ('maskrcnn_resnet50_fpn', 34),
 ('maskrcnn_resnet50_fpn_v2', 35),
 ('quantized_resnet18', 51),
 ('quantized_resnet50', 52),
 ('resnet101', 78),
 ('resnet152', 79),
 ('resnet18', 80),
 ('resnet34', 81),
 ('resnet50', 82),
 ('retinanet_resnet50_fpn', 86),
 ('retinanet_resnet50_fpn_v2', 87),
 ('wide_resnet101_2', 116),
 ('wide_resnet50_2', 117)]

#### Evaluation function

In [9]:

def evaluate(model, criterion):
    with torch.no_grad():
        model.eval()   
        running_loss = 0.0
        running_corrects = 0
        for inputs, labels in dataloaders['val']:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
        epoch_loss = running_loss / dataset_sizes['val']
        epoch_acc = running_corrects.double() / dataset_sizes['val']
    return epoch_loss,epoch_acc

#### Training loop

In [10]:
def train_model(model, criterion, optimizer,scheduler=None, num_epochs=100):

    for epoch in range(num_epochs):
        
        model.train()  
        running_loss = 0.0
        running_corrects = 0
        best_model_wts = copy.deepcopy(model.state_dict())
        best_acc = 0.0
       
        for inputs, labels in dataloaders['train']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
        if scheduler != None:
            scheduler.step()

        epoch_loss = running_loss / dataset_sizes['train']
        epoch_acc = running_corrects.double() / dataset_sizes['train']
        v_loss,v_acc=evaluate(model,criterion)
        writer.add_scalars("loss",{'train':epoch_loss,'valid':v_loss},epoch)
        writer.add_scalars("acc",{'train':epoch_acc,'valid':v_acc},epoch)

        if  v_acc > best_acc:
                best_acc = v_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        if epoch%5==0:
            print(f'Epoch {epoch}/{num_epochs - 1}')
            print('-' * 10)
            
            print("t_loss={:.4f},t_acc={:.4f}".format(epoch_loss,epoch_acc))
            print("v_loss={:.4f},v_acc={:.4f}".format(v_loss,v_acc))
    ## training is done. Return the model with the best validation accuracy
    model.load_state_dict(best_model_wts)
    return model

### Train the last layer only

In [11]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad=False
nfeatures = model.fc.in_features
model.fc = nn.Linear(nfeatures, len(class_names))
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer= optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)
scheduler = lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

In [12]:
from torch.utils.tensorboard import SummaryWriter
import datetime
%load_ext tensorboard
current=datetime.datetime.now()
log_dir = 'logs/tensorboard/' +aug+'-lastFC'+'-'+current.strftime("%c")
writer=SummaryWriter(log_dir)

In [13]:
model= train_model(model, criterion, optimizer,scheduler=scheduler,num_epochs=90)
writer.close()

Epoch 0/89
----------
t_loss=1.0017,t_acc=0.6369
v_loss=0.5841,v_acc=0.8123
Epoch 5/89
----------
t_loss=0.5044,t_acc=0.8199
v_loss=0.3597,v_acc=0.8864
Epoch 10/89
----------
t_loss=0.4511,t_acc=0.8367
v_loss=0.3416,v_acc=0.8760
Epoch 15/89
----------
t_loss=0.4034,t_acc=0.8471
v_loss=0.3152,v_acc=0.9038
Epoch 20/89
----------
t_loss=0.4093,t_acc=0.8538
v_loss=0.3241,v_acc=0.8772
Epoch 25/89
----------
t_loss=0.4004,t_acc=0.8550
v_loss=0.3047,v_acc=0.8980
Epoch 30/89
----------
t_loss=0.4055,t_acc=0.8494
v_loss=0.3062,v_acc=0.8957
Epoch 35/89
----------
t_loss=0.3857,t_acc=0.8642
v_loss=0.3135,v_acc=0.8946
Epoch 40/89
----------
t_loss=0.3560,t_acc=0.8738
v_loss=0.2995,v_acc=0.9027
Epoch 45/89
----------
t_loss=0.3773,t_acc=0.8607
v_loss=0.3044,v_acc=0.8957
Epoch 50/89
----------
t_loss=0.3606,t_acc=0.8703
v_loss=0.2990,v_acc=0.9027
Epoch 55/89
----------
t_loss=0.3855,t_acc=0.8648
v_loss=0.3036,v_acc=0.8899
Epoch 60/89
----------
t_loss=0.3809,t_acc=0.8555
v_loss=0.2983,v_acc=0.8980
E

### Train all layers

In [14]:
model = torchvision.models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#model = torchvision.models.resnet18(weights=None)
nfeatures = model.fc.in_features
model.fc = nn.Linear(nfeatures, len(class_names))
model = model.to(device)
optimizer= optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
scheduler = lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

Create new logger

In [15]:
current=datetime.datetime.now()
log_dir = 'logs/tensorboard/' +aug+'-all'+'-'+current.strftime("%c")
writer=SummaryWriter(log_dir)

In [16]:
criterion = nn.CrossEntropyLoss()
model = train_model(model, criterion, optimizer,scheduler=scheduler,num_epochs=90)

Epoch 0/89
----------
t_loss=0.7982,t_acc=0.6969
v_loss=0.3515,v_acc=0.8749
Epoch 5/89
----------
t_loss=0.1940,t_acc=0.9302
v_loss=0.2007,v_acc=0.9351
Epoch 10/89
----------
t_loss=0.1218,t_acc=0.9600
v_loss=0.1971,v_acc=0.9305
Epoch 15/89
----------
t_loss=0.0842,t_acc=0.9737
v_loss=0.2033,v_acc=0.9258
Epoch 20/89
----------
t_loss=0.0893,t_acc=0.9710
v_loss=0.2277,v_acc=0.9351
Epoch 25/89
----------
t_loss=0.0735,t_acc=0.9725
v_loss=0.2197,v_acc=0.9212
Epoch 30/89
----------
t_loss=0.0609,t_acc=0.9786
v_loss=0.2099,v_acc=0.9328
Epoch 35/89
----------
t_loss=0.0684,t_acc=0.9777
v_loss=0.2070,v_acc=0.9328
Epoch 40/89
----------
t_loss=0.0543,t_acc=0.9820
v_loss=0.2087,v_acc=0.9351
Epoch 45/89
----------
t_loss=0.0552,t_acc=0.9818
v_loss=0.1983,v_acc=0.9409
Epoch 50/89
----------
t_loss=0.0505,t_acc=0.9832
v_loss=0.1932,v_acc=0.9432
Epoch 55/89
----------
t_loss=0.0497,t_acc=0.9826
v_loss=0.2040,v_acc=0.9409
Epoch 60/89
----------
t_loss=0.0506,t_acc=0.9826
v_loss=0.2093,v_acc=0.9386
E

In [17]:
%tensorboard --logdir logs/tensorboard